# Deep Research with a Multi-Agent Workflow

Build an end-to-end research system with the **OpenAI Agents SDK**. The workflow
plans a research strategy, runs web searches concurrently, synthesizes the
findings into a structured Markdown report, and optionally delivers the report
by email.

> **Example query:** *Most popular AI agent frameworks in 2026*

## What you will learn

- Define specialized agents with focused instructions.
- Use structured outputs with Pydantic models.
- Give an agent access to web search and a custom function tool.
- Run independent research tasks concurrently with `asyncio.gather`.
- Orchestrate multiple agents deterministically in Python.
- Inspect the complete workflow with an Agents SDK trace.

## Workflow

```mermaid
flowchart TD
    Q[Research query] --> P[Planner Agent]
    P --> S[Search plan]
    S --> W[Search Agents in parallel]
    W --> R[Search summaries]
    R --> X[Writer Agent]
    X --> M[Structured report]
    M --> E[Email Agent]
    E --> D[Email or push notification]
```

## Agents at a glance

| Agent | Responsibility | Main capability |
|---|---|---|
| Planner | Produces a targeted search plan | Structured output |
| Search | Searches and summarizes one topic | Web search tool |
| Writer | Combines evidence into a detailed report | Structured output |
| Email | Formats and sends the final report | Custom function tool |

---

## 1. Setup and imports

Install the required packages before running the notebook:

```bash
pip install openai-agents pydantic python-dotenv requests
```

Create a `.env` file in the working directory. Keep this file private and never
commit it to Git:

```dotenv
OPENAI_API_KEY=your_openai_api_key

# Required only when USE_EMAIL=True
EMAIL_ADDRESS=you@example.com
EMAIL_SMTP_SERVER=smtp.example.com
EMAIL_APP_PASSWORD=your_email_app_password

# Used as the fallback notification channel when USE_EMAIL=False
PUSHOVER_USER=your_pushover_user_key
PUSHOVER_TOKEN=your_pushover_app_token
```

The imports below provide the Agents SDK primitives, Pydantic schemas,
asynchronous execution, environment-variable loading, and notebook display
helpers.

In [1]:
from agents import Agent, WebSearchTool, trace, Runner, function_tool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio
from IPython.display import display, Markdown


### Load environment variables

`override=True` makes values from `.env` take precedence over variables already
present in the notebook process. This is convenient during development, but
review the behavior before using it in production.

In [2]:
load_dotenv(override=True)

True

### Configure the workflow

- `MODEL_NAME` selects the model used by all four agents.
- `USE_EMAIL=True` sends the report through SMTP.
- `USE_EMAIL=False` uses the Pushover fallback instead.
- `HOW_MANY_SEARCHES` controls the number of research branches. Increasing it
  may improve coverage, but also increases latency and tool usage.

In [3]:
MODEL_NAME = "gpt-5.4-mini"
USE_EMAIL = True
HOW_MANY_SEARCHES = 5

### Configure report delivery

The following helper functions load SMTP and Pushover credentials from the
environment. The email is sent from and to `EMAIL_ADDRESS`; adjust the
recipient logic if reports should go to another address.

> **Security:** Use an app-specific email password where supported. Do not put
> credentials directly in the notebook or commit the `.env` file.

In [4]:
from dotenv import load_dotenv
import requests
import os
import smtplib
from email.message import EmailMessage
load_dotenv(override=True)


EMAIL_ADDRESS = os.getenv("EMAIL_ADDRESS")
EMAIL_SMTP_SERVER = os.getenv("EMAIL_SMTP_SERVER")
EMAIL_APP_PASSWORD = os.getenv("EMAIL_APP_PASSWORD")

def send_email(subject, text_body, html_body):
    msg = EmailMessage()
    msg["From"] = EMAIL_ADDRESS
    msg["To"] = EMAIL_ADDRESS
    msg["Subject"] = subject
    msg.set_content(text_body)
    msg.add_alternative(html_body, subtype="html")

    with smtplib.SMTP(EMAIL_SMTP_SERVER, 587) as server:
        server.starttls()
        server.login(EMAIL_ADDRESS, EMAIL_APP_PASSWORD)
        server.send_message(msg)


pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

## 2. Search Agent

The Search Agent handles **one search topic at a time**. Its instructions keep
each result concise so the Writer Agent receives focused evidence rather than
unfiltered search output.

`tool_choice="required"` ensures that the agent actually invokes web search
instead of answering only from model knowledge.

In [8]:
INSTRUCTIONS = """
You are a research assistant. Given a search term, you search the web for that term and 
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 words.
Capture the main points and be succinct. Reply only with the summary.
"""
task = "Most popular AI Agent frameworks in 2026"

settings = ModelSettings(tool_choice="required")
tools = [WebSearchTool()]

In [9]:
search_agent = Agent(name="Search Agent", instructions=INSTRUCTIONS, tools=tools, model=MODEL_NAME, model_settings=settings)

### Test the Search Agent

Run a single search first to verify the model, API key, and web-search tool
before executing the complete multi-agent workflow.

In [10]:
result = await Runner.run(search_agent, task)
display(Markdown(result.final_output))

In 2026, the most widely discussed AI agent frameworks are **LangGraph/LangChain, CrewAI, Microsoft AutoGen, Dify, and OpenAI’s Agents SDK**. Across recent roundups and comparisons, **LangGraph** is often positioned as the strongest general-purpose choice for production agents and complex stateful workflows, **CrewAI** for role-based multi-agent collaboration, and **AutoGen** for enterprise and conversational multi-agent patterns. GitHub-based 2026 comparisons also consistently place these tools near the top by ecosystem breadth, usage, or stars. ([github.com](https://github.com/larsderidder/framework-analysis?utm_source=openai))

If you’re looking for the “most popular” by developer attention, the landscape is broader than just frameworks: **Dify** appears especially strong as a popular open-source agent platform, while **LangGraph** remains the most cited framework for building resilient production agents. Other frameworks gaining traction in 2026 include **Pydantic AI, Vercel AI SDK, Letta, Mastra, Agno, and Microsoft Agent Framework**. Enterprise adoption is accelerating overall, with a 2026 CrewAI survey reporting that 65% of enterprises are already using AI agents and 100% plan to expand adoption this year. ([dify.ai](https://dify.ai/about-us?utm_source=openai))

## 3. Planner Agent

The Planner Agent converts the broad research question into a fixed number of
targeted searches. Pydantic models define the output contract:

- `query` contains the precise web-search phrase.
- `reason` explains how that search contributes to the final answer.
- `WebSearchPlan` groups all planned searches in one validated object.

Structured output makes the plan predictable and easy to consume from Python.

In [11]:
class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")
    query: str = Field(description="The search term to use for the web search.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")

### Inspect the structured-output schema

This cell displays the JSON Schema generated by Pydantic. The SDK uses this
schema to constrain the Planner Agent's response.

In [12]:
WebSearchPlan.model_json_schema()

{'$defs': {'WebSearchItem': {'properties': {'reason': {'description': 'Your reasoning for why this search is important to the query.',
     'title': 'Reason',
     'type': 'string'},
    'query': {'description': 'The search term to use for the web search.',
     'title': 'Query',
     'type': 'string'}},
   'required': ['reason', 'query'],
   'title': 'WebSearchItem',
   'type': 'object'}},
 'properties': {'searches': {'description': 'A list of web searches to perform to best answer the query.',
   'items': {'$ref': '#/$defs/WebSearchItem'},
   'title': 'Searches',
   'type': 'array'}},
 'required': ['searches'],
 'title': 'WebSearchPlan',
 'type': 'object'}

### Create and test the Planner Agent

The number of requested searches is injected from `HOW_MANY_SEARCHES`. The
returned value is a `WebSearchPlan`, not an unstructured text response.

In [13]:
# See note above about cost of WebSearchTool

INSTRUCTIONS = f"""
You are a research assistant. Given a user query, come up with a set of web searches
to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for.
"""

planner_agent = Agent(name="Planner Agent", instructions=INSTRUCTIONS, model=MODEL_NAME, output_type=WebSearchPlan)

In [15]:

result = await Runner.run(planner_agent, task)
result.final_output

WebSearchPlan(searches=[WebSearchItem(reason='Identify current rankings, adoption, and developer mindshare for AI agent frameworks in 2026.', query='most popular AI agent frameworks 2026 ranking'), WebSearchItem(reason='Find comparative analyses and benchmark-style articles covering leading agent frameworks and their usage.', query='AI agent frameworks comparison 2026 LangGraph CrewAI AutoGen Semantic Kernel'), WebSearchItem(reason='Check recent GitHub stars, releases, and community activity for framework popularity signals.', query='GitHub stars AI agent frameworks 2026 LangGraph CrewAI AutoGen'), WebSearchItem(reason='See which frameworks are recommended in recent industry reports, surveys, or blog posts.', query='2026 AI agent framework survey adoption report'), WebSearchItem(reason='Capture emerging frameworks and alternatives that may have become popular by 2026.', query='new AI agent frameworks 2026 popular alternatives')])

## 4. Writer Agent

The Writer Agent receives the original question and all search summaries, then
synthesizes them into one coherent report. Its structured response separates:

1. a short executive summary,
2. the full Markdown report, and
3. suggested follow-up questions.

This separation makes the output reusable in a UI, API, database, or email
pipeline.

In [16]:
INSTRUCTIONS = """
You are a senior researcher tasked with writing a cohesive report for a research query.
You will be provided with the original query, and some research.
Generate a comprehensive report based on the research and the query.
The final output should be in markdown format, and it should be lengthy and detailed. Aim 
for 5-10 pages of content, at least 1000 words.
"""


class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")
    markdown_report: str = Field(description="The final report")
    follow_up_questions: list[str] = Field(description="Suggested topics to research further")


writer_agent = Agent(name="Writer Agent", instructions=INSTRUCTIONS, model=MODEL_NAME, output_type=ReportData)

## 5. Email Agent and custom tool

The Email Agent turns the Markdown report into a polished HTML email and calls
`send_email_tool`. Marking the Python function with `@function_tool` exposes
its typed parameters to the agent.

When `USE_EMAIL=False`, the same tool sends a shorter Pushover notification.
This makes it possible to test the workflow without sending an email.

In [17]:
@function_tool
def send_email_tool(subject: str, text_body: str, html_body: str) -> str:
    """
    Send out an email with the given subject and body to all sales prospects
    
    Args:
        subject: The subject of the email
        text_body: The body of the email as plain text
        html_body: The HTML body of the email
    """
    if USE_EMAIL:
        send_email(subject, text_body, html_body)
    else:
        push(f"Subject: {subject}\n\n{text_body}")
    return "Email sent successfully"

### Inspect the function-tool schema

The generated schema shows exactly which arguments the Email Agent must supply
when it invokes the custom tool.

In [18]:
send_email_tool.params_json_schema

{'properties': {'subject': {'description': 'The subject of the email',
   'title': 'Subject',
   'type': 'string'},
  'text_body': {'description': 'The body of the email as plain text',
   'title': 'Text Body',
   'type': 'string'},
  'html_body': {'description': 'The HTML body of the email',
   'title': 'Html Body',
   'type': 'string'}},
 'required': ['subject', 'text_body', 'html_body'],
 'title': 'send_email_tool_args',
 'type': 'object',
 'additionalProperties': False}

In [19]:
INSTRUCTIONS = """
You are provided with a detailed report. Use your tool to send an email, converting the report into
a clean, well presented HTML email with an appropriate subject line.
"""

email_agent = Agent(name="Email Agent", instructions=INSTRUCTIONS, tools=[send_email_tool], model=MODEL_NAME)

## 6. Orchestrate the agents in Python

The orchestration is deterministic and easy to inspect:

1. `run_searches()` asks the Planner Agent for a search plan.
2. Each plan item is passed to `search()`.
3. `asyncio.gather()` executes independent searches concurrently.
4. `write_report()` sends the collected summaries to the Writer Agent.
5. `send_report_email()` asks the Email Agent to deliver the report.

This pattern combines **LLM-based decisions inside each agent** with
**code-based control over the overall workflow**.

In [21]:
async def run_searches(query: str):
    print("Planning searches...")
    result = await Runner.run(planner_agent, f"Query: {query}")
    searches = result.final_output.searches
    print(f"Will perform {len(searches)} searches")
    tasks = [search(item) for item in searches]
    results = await asyncio.gather(*tasks)
    print("Finished searching")
    return results
    
    
async def search(item: WebSearchItem):
    input_message = f"Search term: {item.query}\nReason for searching: {item.reason}"
    result = await Runner.run(search_agent, input_message)
    return result.final_output

In [22]:
async def write_report(query: str, search_results: list[str]):
    print("Thinking about report...")
    input_message = f"Original query: {query}\nSummarized search results: {search_results}"
    result = await Runner.run(writer_agent, input_message)
    print("Finished writing report")
    return result.final_output

async def send_report_email(report: ReportData):
    print("Writing email...")
    result = await Runner.run(email_agent, report.markdown_report)
    print("Email sent")
    return result.final_output

## 7. Run the complete deep-research workflow

Change `query` to research another topic. The `trace()` context groups all agent
and tool activity under one trace, which is useful for debugging latency,
instructions, handoffs, and tool calls.

> **Before running:** Set `USE_EMAIL=False` if you want to test without sending
> an email.

In [23]:
query ="Most popular AI Agent frameworks in 2026"

with trace("Research trace"):
    print("Starting research...")
    search_results = await run_searches(query)
    report = await write_report(query, search_results)
    await send_report_email(report)  
    print("Hooray!")

Starting research...
Planning searches...
Will perform 5 searches
Finished searching
Thinking about report...
Finished writing report
Writing email...
Email sent
Hooray!


## 8. Inspect the final report

After the workflow completes, use the following optional cell to render the
executive summary, full report, and suggested follow-up questions directly in
the notebook:

```python
display(Markdown(f"## Executive summary\n\n{report.short_summary}"))
display(Markdown(report.markdown_report))
display(Markdown(
    "## Follow-up questions\n\n"
    + "\n".join(f"- {question}" for question in report.follow_up_questions)
))
```

## Ideas for extending the project

- Add source URLs and citations to every major claim.
- Introduce retries, timeouts, and exception handling for network operations.
- Add a reviewer agent that checks evidence quality before delivery.
- Save reports to a database or object store.
- Stream progress updates to a frontend.
- Give each search task a concurrency limit for larger research plans.
- Add automated evaluations for factuality, coverage, and citation quality.

## Production considerations

- Validate all required environment variables at startup.
- Check HTTP and SMTP responses and use explicit timeouts.
- Log failures without exposing secrets.
- Review generated reports before using them for high-stakes decisions.
- Treat web content as untrusted input and guard against prompt injection.

---

### Key takeaway

This notebook demonstrates a practical multi-agent architecture: the Planner
decides **what to investigate**, Search Agents gather evidence **in parallel**,
the Writer turns that evidence into a structured deliverable, and the Email
Agent handles **delivery through a tool**.